### Using Softmax layer with Cross Entropy Loss

In [2]:
import torch 
import torch.nn as nn 
import numpy as np 

#### Softmax 
- With more than one element output, we use take the exponential of each element and divide the sum to get the probability of each output.  
- The element with the highest probability is the softmax layer output 
- Good at classification tasks


In [3]:
def softmax(x): 
    return np.exp(x)/np.sum(np.exp(x), axis=0)  # exponential each element and divide the sum of the exponential. Sum is performed on rows 

In [6]:
x=np.array([2.0,1.0,0.1])
output= softmax(x)
print("Probability of individual output",output)   #total will add up to be 1 
print("Combined Probability: ",np.sum(output))

Probability of individual output [0.65900114 0.24243297 0.09856589]
Combined Probability:  1.0


In [7]:
#PyTorch implementation 
x= torch.tensor([2.0,1.0,0.1])
output= torch.softmax(x,dim=0)  #compute the sum along the rows 
print(output)

tensor([0.6590, 0.2424, 0.0986])


#### Cross Entropy 
sum of the ground truth * log(y_prediction) * -1    
  
The further apart prediction is to the ground truth, the higher the loss is.  
- Because cross entropy is based on probability distributions where p(x) is the probability distribution of the correct label and log(p(x)) is the probability of prediction. It is best used when the prediction and ground truth is $\in [0,1]$

In [10]:
def cross_entropy(y_hat, y):
    loss= -np.sum(y * np.log(y_hat))
    return loss 

Y= np.array([1,0,0])
y1= np.array([0.7,0.2,0.1])
y2 =np.array([0.1,0.3,0.6])
l1= cross_entropy(y1,Y)
l2= cross_entropy(y2,Y)

print (f"First loss: {l1:.6f}")
print (f"Second loss: {l2:.6f}")

First loss: 0.356675
Second loss: 2.302585


#### PyTorch CrossEntropy Loss

softmax + negative log likehood loss

In [12]:
loss = nn.CrossEntropyLoss()

Y= torch.tensor([0])    
y_good= torch.tensor([[2.0,1.0,0.1]])# n_samples x n_classes. 

# Pytorch will apply softmax layer compute the loss sum with the ground truth 
y_bad= torch.tensor([[0.5,2.0,0.3]])

l1= loss(y_good,Y)
l2= loss(y_bad, Y)
print (f"Good loss: {l1.item():.6f}")
print (f"Loss loss: {l2.item():.6f}")

Good loss: 0.417030
Loss loss: 1.840616


In [1]:
import torch 

In [8]:
a=torch.tensor([3, 3.1,3.5,2.9], dtype=torch.float32)
temperature = 0.9
print(torch.softmax(a/temperature, dim=-1))

tensor([0.2103, 0.2350, 0.3665, 0.1882])


In [27]:
a=torch.tensor([10,10.5,10.1,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9,0.9], dtype=torch.float32)
temperature = 1
print(torch.softmax(a/temperature, dim=-1))

tensor([2.6609e-01, 4.3871e-01, 2.9407e-01, 2.9713e-05, 2.9713e-05, 2.9713e-05,
        2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05,
        2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05,
        2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05,
        2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05,
        2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05,
        2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05, 2.9713e-05])


### Softmax at small vs. Large Elements: Does small logtis difference trigger large probability difference? 

#### Similar numbers: 
When numbers are similar with each other, the difference in probability with the rest will become smaller as the number of elements go up 
- this is intuitive in a sense that your individual probability becomes smaller and the sum becomes greater 

In [1]:
import numpy as np 
def softmax(x):
    # Subtracting the max for numerical stability
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()


case1 = np.array([3.0, 3.2, 3.0])
print(f"Case 1 (3 classes): {softmax(case1)}")

case2 = np.array([3.2] + [3.0] * 100)
probs2 = softmax(case2)
print(f"Case 2 (101 classes): 3.2 prob = {probs2[0]:.4f}, single 3 prob = {probs2[1]:.4f}")

case3 = np.array([3.2] + [3.0] * 1000)
probs3 = softmax(case3)
print(f"Case 3 (1001 classes): 3.2 prob = {probs3[0]:.4f}, single 3 prob = {probs3[1]:.4f}")


Case 1 (3 classes): [0.31042377 0.37915245 0.31042377]
Case 2 (101 classes): 3.2 prob = 0.0121, single 3 prob = 0.0099
Case 3 (1001 classes): 3.2 prob = 0.0012, single 3 prob = 0.0010


### What about if the winner is very different from the rest of logits? Consider only one winner case 

- small differences in logits do influence the final softmax outcome by quite large margin. But the bigger the difference, the bigger the probability difference, and vice versa. So loss function can still penalize. 

In [7]:
case1 = np.array([3.2, 1.2,1.2])
print(f"Case 1 (3 classes): {softmax(case1)}")

case2 = np.array([3.2] + [1.2] * 100)
probs2 = softmax(case2)
print(f"Case 2 (101 classes): 3.2 prob = {probs2[0]:.4f}, 1.2 prob = {probs2[1]:.4f}")

case3 = np.array([3.2] + [1.2] * 1000)
probs3 = softmax(case3)
print(f"Case 3 (1001 classes): 3.2 prob = {probs3[0]:.4f}, 1.2 prob = {probs3[1]:.4f}")

Case 1 (3 classes): [0.78698604 0.10650698 0.10650698]
Case 2 (101 classes): 3.2 prob = 0.0688, 1.2 prob = 0.0093
Case 3 (1001 classes): 3.2 prob = 0.0073, 1.2 prob = 0.0010


### Consider multiple winner case, such as 2
- their logits difference does split the probability difference. The closer they are, the closer their probability is. The larger the logits difference, the larger probability difference. 
- all depends on the relative distance. the larger the winner and closer

In [23]:
case1 = np.array([3.2, 2.2,1.2])
print(f"Case 1 (3 classes): {softmax(case1)}")

case2 = np.array([3.2] + [2.9] + [1.2] * 100)
probs2 = softmax(case2)
print(f"Case 2 (101 classes): 3.2 prob = {probs2[0]:.4f}, 2.9 prob = {probs2[1]:.4f}, 1.2 prob = {probs2[2]:.4f}")

case3 = np.array([10.9] + [2.9] + [1.2] * 100000)
probs3 = softmax(case3)
print(f"Case 3 (1001 classes): 3.2 prob = {probs3[0]:.4f}, 2.9 prob = {probs3[1]:.4f}, 1.2 prob = {probs3[2]:.4f}")

Case 1 (3 classes): [0.66524096 0.24472847 0.09003057]
Case 2 (101 classes): 3.2 prob = 0.0655, 2.9 prob = 0.0485, 1.2 prob = 0.0089
Case 3 (1001 classes): 3.2 prob = 0.1403, 2.9 prob = 0.0000, 1.2 prob = 0.0000


## Conclusion 
- differences in small logits will result in small differences in probability, so numerically it's fine. it won't cause vanishing and exploding gradients, not the main concern
- the real reaosn to use hiearchical softmax or negative sampling is for speedup. 